# 딥러닝 입문 — 프로젝트 Starter 노트북

## 팀 정보 (여기를 먼저 채우세요)

- **N반 / 팀 번호 10**:
- **팀원 및 역할**:
  - 임동욱: 담당 (예: Augmentation 실험)
  - 이현학: 담당
- **담당 실험 주제**: (AUG / ARCH / OPT / RES / TL-FE / TL-FT 중)
- **베이스라인 최종 성능**: (나중에 기록)

---

> 이 노트북은 **데이터 로딩 + SimpleCNN 정의 + 학습 루프 골격**까지 제공합니다.
> 학습 루프의 핵심 라인(`______` 표시)과 평가·실험은 여러분이 직접 채웁니다.
> 6~10주차 노트북을 참고하세요.

### ⚠️ 시작 전 필수: GPU 켜기
`런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU → 저장`
GPU 없이 돌리면 1 epoch에 수십 분 걸립니다. 아래 셀에서 `cuda` 가 찍히는지 꼭 확인하세요.


---
## 0. 환경 설정

In [ ]:
# 라이브러리 임포트
import os
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms, datasets

# 재현성 시드 고정
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# 디바이스 확인 (cuda 가 나와야 정상)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ GPU가 아닙니다! 런타임 유형을 T4 GPU로 변경 후 처음부터 다시 실행하세요.')


---
## 1. 데이터 다운로드 (PlantVillage)

PlantVillage Color 데이터셋 (38 클래스, 약 54,000장)을 **gdown**으로 받습니다.
Kaggle 계정이 필요 없습니다. 이 셀은 처음 한 번만 다운로드하고, 이후에는 캐시를 재사용합니다.

> 압축 해제 후 클래스 폴더 위치는 자동으로 찾습니다.
> 혹시 못 찾으면 폴더 트리가 출력되니, 안내대로 `IMAGE_ROOT` 한 줄만 직접 지정하세요.

In [ ]:
!pip install -q gdown
import gdown

# 강사가 배포한 공통 데이터 (전체 팀 동일하게 사용)
GDRIVE_FILE_ID = '1oOe-OAYWOneBxADfABdZ8MVc7kq6zriO'

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)
ZIP_PATH = DATA_DIR / 'plantvillage.zip'

# 클래스 폴더 탐지 헬퍼 (확장자 대소문자 무관)
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')

def has_images(folder):
    try:
        for f in os.listdir(folder):
            if f.lower().endswith(IMG_EXTS):
                return True
    except (NotADirectoryError, PermissionError):
        pass
    return False

def find_image_root(start):
    # 하위에 '이미지를 직접 담은 폴더'가 10개 이상인 디렉토리를 클래스 루트로 판단
    for root, dirs, files in os.walk(start):
        subdirs = [d for d in dirs if not d.startswith('.')]
        class_like = [d for d in subdirs if has_images(os.path.join(root, d))]
        if len(class_like) >= 10:
            return Path(root)
    return None

def print_tree(start, max_depth=3):
    start = Path(start)
    base = len(start.parts)
    for root, dirs, files in os.walk(start):
        depth = len(Path(root).parts) - base
        if depth > max_depth:
            dirs[:] = []
            continue
        n_img = sum(1 for f in files if f.lower().endswith(IMG_EXTS))
        tag = f'  [이미지 {n_img}개]' if n_img else ''
        print('  ' * depth + f'{Path(root).name}/  (하위폴더 {len(dirs)}개){tag}')

# 데이터가 이미 있으면 다운로드/압축해제 스킵 (압축 위치 무관 자동 판단)
IMAGE_ROOT = find_image_root(DATA_DIR)

if IMAGE_ROOT is None:
    if not ZIP_PATH.exists():
        url = f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}'
        gdown.download(url, str(ZIP_PATH), quiet=False)
    print('압축 해제 중...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)
    print('완료')
    IMAGE_ROOT = find_image_root(DATA_DIR)
else:
    print('데이터가 이미 있습니다. 다운로드/압축해제 스킵')

if IMAGE_ROOT is None:
    print('❌ 클래스 폴더를 자동으로 못 찾았습니다. 압축 해제 구조:')
    print('-' * 60)
    print_tree(DATA_DIR, max_depth=3)
    print('-' * 60)
    print("위 트리에서 클래스 폴더(예: Apple___Black_rot)들이 직접 들어있는")
    print("디렉토리를 찾아 아래처럼 지정 후 이 셀을 다시 실행하세요. 예:")
    print("  IMAGE_ROOT = Path('./data/color')")
    raise SystemExit('IMAGE_ROOT 수동 지정 필요')

print(f'이미지 루트 경로: {IMAGE_ROOT}')
print(f'클래스 수       : {len(list(IMAGE_ROOT.iterdir()))}')


---
## 2. Dataset / DataLoader

### transform 분리 원칙 (중요)
- **train transform**: 학습용. **Augmentation 실험 팀은 여기를 수정**합니다.
- **eval transform**: val/test 공통. **절대 augmentation을 넣지 마세요** (평가가 오염됨).

> **왜 `TransformedSubset`을 쓰나요?**
> `random_split`은 원본 데이터셋 하나를 공유합니다. 그래서 `val_set.dataset.transform = ...`
> 처럼 바꾸면 **train_set의 transform까지 같이 바뀌는 버그**가 생깁니다.
> `TransformedSubset`은 분할은 그대로 두고 transform만 따로 입혀 이 문제를 막습니다.
> AUG 팀이 train transform을 바꿔도 val/test는 안전합니다.

In [ ]:
# PlantVillage(color) 통계값. Transfer Learning 팀은 ImageNet 통계로 바꾸세요:
#   IMAGENET_MEAN=[0.485,0.456,0.406], IMAGENET_STD=[0.229,0.224,0.225]
PLANT_MEAN = (0.4717, 0.4963, 0.3978)
PLANT_STD  = (0.1951, 0.1668, 0.1948)
IMG_SIZE = 128

# 평가용 transform (val/test 공통) - augmentation 절대 금지
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(PLANT_MEAN, PLANT_STD),
])

# 학습용 transform (베이스라인은 augmentation 없음)
# ───────────────────────────────────────────────────────────
# [AUG 실험 팀] 아래 Compose 안에 augmentation을 추가하세요.
# ⚠️ 데이터가 클래스당 150장으로 작습니다. 강한 변형(회전 큰 각도,
#    ColorJitter 3종 동시, 강한 색 변화)은 15 epoch 안에 수렴을 못 해
#    오히려 성능이 떨어집니다. '약하게' 시작해 조금씩 키우며 비교하세요.
# 약한 예시(권장 출발점):
#   transforms.RandomHorizontalFlip(p=0.5),
#   transforms.RandomRotation(10),
#   transforms.ColorJitter(brightness=0.1, contrast=0.1),
# 주의: 잎은 위/아래 구분이 있으므로 VerticalFlip은 신중히.
#       강도를 키울 거면 EPOCHS도 함께 늘려 공정 비교 조건을 맞추세요.
# ───────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    # <-- AUG 팀: 이 줄들 사이에 augmentation 추가
    transforms.ToTensor(),
    transforms.Normalize(PLANT_MEAN, PLANT_STD),
])


In [ ]:
# transform을 안전하게 분리해주는 래퍼
# 중첩 구조(random_split -> Subset(클래스축소) -> ImageFolder)를 끝까지 풀어
# 원본 이미지를 직접 읽고 split마다 다른 transform을 안전하게 적용한다.
class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def _resolve(self, idx):
        dataset = self.subset
        while isinstance(dataset, torch.utils.data.Subset):
            idx = dataset.indices[idx]
            dataset = dataset.dataset
        return dataset, idx

    def __getitem__(self, idx):
        image_folder, abs_idx = self._resolve(idx)
        path, label = image_folder.samples[abs_idx]
        img = image_folder.loader(path)        # 원본 PIL 이미지
        if self.transform is not None:
            img = self.transform(img)
        return img, label


# 원본 데이터셋
full_dataset = datasets.ImageFolder(str(IMAGE_ROOT), transform=None)
CLASSES = full_dataset.classes
NUM_CLASSES = len(CLASSES)

# ── 클래스당 이미지 수 제한 (난이도 조절) ───────────────────────
# PlantVillage 전체를 쓰면 baseline조차 95%+가 나와 실험 간 차이가 안 보입니다.
# 클래스당 일부만 사용해 변별력을 확보합니다.
# ⚠️ PER_CLASS와 SEED는 모든 팀이 동일하게 두세요 (공정 비교의 전제).
PER_CLASS = 150

import random as _rnd
from collections import defaultdict
from torch.utils.data import Subset
_rnd.seed(SEED)

_by_class = defaultdict(list)
for idx, label in enumerate(full_dataset.targets):
    _by_class[label].append(idx)
selected = []
for label, idxs in _by_class.items():
    _rnd.shuffle(idxs)
    selected.extend(idxs[:PER_CLASS])
_rnd.shuffle(selected)
base_dataset = Subset(full_dataset, selected)

print(f'Num classes : {NUM_CLASSES}')
print(f'클래스당 제한: {PER_CLASS}장')
print(f'사용 이미지  : {len(base_dataset):,}  (원본 {len(full_dataset):,})')

# 70 / 15 / 15 분할 (강사 참고본과 동일 비율·시드)
N = len(base_dataset)
n_train = int(N * 0.70)
n_val   = int(N * 0.15)
n_test  = N - n_train - n_val
g = torch.Generator().manual_seed(SEED)
train_set, val_set, test_set = random_split(base_dataset, [n_train, n_val, n_test], generator=g)

train_ds       = TransformedSubset(train_set, train_transform)
train_eval_ds  = TransformedSubset(train_set, eval_transform)   # train acc 측정용(증강 X)
val_ds         = TransformedSubset(val_set,   eval_transform)
test_ds        = TransformedSubset(test_set,  eval_transform)

# GPU 없으면 pin_memory 경고가 나므로 자동 분기
PIN = torch.cuda.is_available()
BATCH_SIZE = 64

train_loader      = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               num_workers=2, pin_memory=PIN)
train_eval_loader = DataLoader(train_eval_ds, batch_size=128, shuffle=False,
                               num_workers=2, pin_memory=PIN)
val_loader        = DataLoader(val_ds,   batch_size=128, shuffle=False,
                               num_workers=2, pin_memory=PIN)
test_loader       = DataLoader(test_ds,  batch_size=128, shuffle=False,
                               num_workers=2, pin_memory=PIN)

print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}')


In [ ]:
# 샘플 이미지 시각화 (normalize 되돌려서 표시)
inv_mean = torch.tensor(PLANT_MEAN).view(3, 1, 1)
inv_std  = torch.tensor(PLANT_STD).view(3, 1, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i, ax in enumerate(axes.flat):
    img = images[i] * inv_std + inv_mean
    ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(CLASSES[labels[i]][:18], fontsize=7)
    ax.axis('off')
plt.suptitle('PlantVillage Sample Images', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# 클래스 분포 확인 (불균형 정도 파악 - OPT/ARCH 팀 참고)
label_counts = Counter([label for _, label in full_dataset.samples])
counts = [label_counts[i] for i in range(NUM_CLASSES)]

plt.figure(figsize=(16, 4))
plt.bar(range(NUM_CLASSES), counts, color='steelblue')
plt.xticks(range(NUM_CLASSES), [c[:15] for c in CLASSES], rotation=90, fontsize=6)
plt.ylabel('Num images'); plt.title('Class Distribution')
plt.tight_layout(); plt.show()
print(f'Min: {min(counts)} | Max: {max(counts)} | Mean: {np.mean(counts):.0f}')


---
## 3. 모델 정의 — SimpleCNN (Baseline)

입력: 3 × 128 × 128 → 출력: 38개 클래스

- **ARCH(모델 구조) 팀**: 이 클래스를 수정하거나 새 클래스를 추가하세요.
- **RES(ResNet) 팀**: 10주차 `ResidualBlock`/`DownsampleBlock` 코드를 가져와 새 모델을 만드세요.
- **TL-FE / TL-FT 팀**: `torchvision.models.resnet18(weights=...)` 를 불러와 fc를 38로 교체하세요.
  (이때 normalize는 ImageNet 통계로 — Section 2 주석 참고)

In [ ]:
class SimpleCNN(nn.Module):
    # PlantVillage 베이스라인 CNN. [Conv-BN-ReLU-Pool] x3 -> FC
    # 공간: 128 -> 64 -> 32 -> 16,  채널: 3 -> 32 -> 64 -> 128
    def __init__(self, num_classes=38):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                       # 128 -> 64

            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                       # 64 -> 32

            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),                       # 32 -> 16
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)   # flatten
        return self.classifier(x)


# Sanity check
model = SimpleCNN(num_classes=NUM_CLASSES).to(device)
print(f'SimpleCNN total parameters: {sum(p.numel() for p in model.parameters()):,}')
with torch.no_grad():
    dummy = torch.randn(4, 3, 128, 128).to(device)
    print(f'Output shape: {model(dummy).shape}   (4, 38 이어야 정상)')


---
## 4. 학습 루프 — 여기서부터 직접 작성

아래는 **골격과 5단계 가이드**입니다. `______` (언더바 6개) 부분을 채우세요.
3주차에서 배운 학습 5단계를 떠올리세요:

```
zero_grad()  →  forward  →  loss 계산  →  backward()  →  step()
```

> **체크포인트**: `model.train()` / `model.eval()` 호출 위치(6주차 Dropout/BN),
> `torch.no_grad()` 사용(3주차), `loss.item()` 누적 시 배치 크기 가중 평균 — 모두 신경 쓰세요.

> ⚠️ **train accuracy 측정 주의 (특히 AUG 팀)**
> 학습 루프 *안에서* 정확도를 재면 ① augmentation으로 변형된 이미지 ② Dropout 켜진
> 상태라서 실제보다 훨씬 낮게 나옵니다(예: train 25% vs val 63% 같은 착시).
> 그래서 아래 골격은 **학습은 train_loader로(가중치 갱신만), train 정확도는
> `train_eval_loader`(증강 없는 깨끗한 이미지)로 다시 측정**하도록 짜여 있습니다.
> 이렇게 해야 val과 같은 기준이 되어 과적합/과소적합을 올바로 진단할 수 있습니다.

> ⚠️ **epoch 부족 주의**: augmentation을 강하게 쓰면 더 많은 epoch가 필요합니다.
> 모든 실험을 같은 EPOCHS로 비교하되, val 곡선이 끝까지 우상향이면 수렴 전이니
> EPOCHS를 늘리세요(예: 30). 강사 참고 설정은 30입니다.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    # 가중치 업데이트만 담당 (정확도는 여기서 신뢰하지 말 것)
    model.______()                       # 학습 모드 (Dropout/BN 활성)
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # ── 학습 5단계 ──────────────────────────────
        optimizer.______()               # 1) 이전 gradient 초기화
        logits = ______                  # 2) forward
        loss = ______                    # 3) loss 계산 (criterion 사용)
        loss.______()                    # 4) backward
        optimizer.______()               # 5) 파라미터 업데이트
        # ────────────────────────────────────────────


@torch.no_grad()                          # 평가 중 gradient 계산 불필요
def evaluate(model, loader, criterion):
    # train_eval_loader / val_loader / test_loader 모두 이 함수로 측정
    model.______()                       # 평가 모드
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total


In [ ]:
# 전체 학습 실행 — 직접 완성하세요
EPOCHS = 30                               # 모든 실험 동일하게 (강사 참고 설정과 동일)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for ep in range(1, EPOCHS + 1):
    train_one_epoch(model, train_loader, optimizer, criterion)   # 학습(가중치 갱신)
    tr_loss, tr_acc = ______              # evaluate(... train_eval_loader ...) 호출
    va_loss, va_acc = ______              # evaluate(... val_loader ...) 호출
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(va_loss)
    history['val_acc'].append(va_acc)
    print(f'Epoch {ep:2d}/{EPOCHS} | '
          f'train acc {tr_acc*100:5.2f}% | val acc {va_acc*100:5.2f}%')

# 마지막에 test_loader로 최종 성능도 측정해 보세요
# test_loss, test_acc = evaluate(model, test_loader, criterion)
# print(f'Test accuracy: {test_acc*100:.2f}%')


In [ ]:
# Train / Val accuracy 그래프 — 직접 완성하세요
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ep_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(ep_range, history['train_loss'], 'o-', label='train')
axes[0].plot(ep_range, ______, 'o-', label='val')        # val_loss
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep_range, [a*100 for a in history['train_acc']], 'o-', label='train')
axes[1].plot(ep_range, ______, 'o-', label='val')        # val_acc (단위 %로)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


---
## 5. 실험 섹션 (팀원별 담당 명시)

베이스라인이 돌아간 뒤, 담당 주제로 실험을 추가하세요.
각 실험은 **별도 섹션 + 담당자 표기 + 베이스라인 대비 비교 그래프**로 작성합니다.

| 주제 코드 | 무엇을 바꾸나 | 어디를 수정 |
|---|---|---|
| AUG | augmentation 조합·강도 | Section 2 `train_transform` |
| ARCH | 채널 수·depth·구조 | Section 3 SimpleCNN |
| OPT | optimizer / lr / scheduler | Section 4 optimizer |
| RES | ResidualBlock 도입 (10주차) | Section 3 새 모델 |
| TL-FE | 사전학습 ResNet, fc만 학습 | Section 3 + normalize 변경 |
| TL-FT | 사전학습 ResNet, 단계적 unfreeze | Section 3 + normalize 변경 |

> 공정 비교 원칙: **한 번에 한 가지만** 바꾸고 EPOCHS·batch size 등 나머지는 고정.

예시 섹션 제목:
```
## 실험 2: Augmentation 추가 (담당: 홍길동)
```


---
## 발표 요약 (13주차 전에 작성)

- **시도한 실험 (3가지 이상)**:
  1.
  2.
  3.

- **베이스라인 대비 최종 성능**:

- **가장 효과적이었던 것과 이유**:

- **아쉬웠던 점 / 더 해보고 싶었던 것**:

- **다른 팀에게 추천**:

---
### 평가 배점 (절대평가, 참고용)
실험 과정 40 + 코드 품질 30 + 발표 20 + 성능 10 = 100점
(A: 85+, B: 70~84, C: 55~69, D: 55 미만)
